# Appendix: tf.data and GradientTape

The main path of section 05 runs Keras 3 on PyTorch. Two TensorFlow features have no direct Keras 3 equivalent, so they live here: `tf.data` for input pipelines and `tf.GradientTape` for a hand-written training loop. It needs the `tf` extra, and TensorFlow cannot share an environment with `umap` (both load their own LLVM, and the kernel aborts). Use a second environment: `UV_PROJECT_ENVIRONMENT=.venv-tf uv sync --extra torch --extra tf`. The main test suite skips this notebook.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

## Intuition

A model is only as fast as its input pipeline. `tf.data` describes the pipeline as a graph of steps (shuffle, map, batch, prefetch) that TensorFlow runs in parallel with training. `GradientTape` records operations as they run, then replays them backwards to get gradients. It is the same idea as the autodiff you built in section 01, with the tape made explicit.

Import order matters here. TensorFlow and PyTorch can share a process, but loading MNIST (which uses torchvision) after TensorFlow aborts the kernel with an LLVM error on this setup. So the data loads and the seed is set first, and TensorFlow comes last.

In [ ]:
import numpy as np

from common.datasets import load_mnist
from common.dl import set_seed, setup_keras

set_seed(42)
x_train, y_train, x_test, y_test = load_mnist(n_train=2000, n_test=500)

setup_keras("tensorflow")
import keras
import tensorflow as tf

tf.random.set_seed(42)
print(keras.backend.backend(), x_train.shape)

## Maths and cost function

The loss is cross-entropy, $L = -\frac{1}{n}\sum_i \log p_{i, y_i}$, and the update is plain gradient descent, $w \leftarrow w - \eta \, \nabla_w L$. Nothing new. The new part is how the data arrives and how the gradient is taken.

## tf.data pipeline

Build a pipeline in four steps. Shuffle with a fixed seed, batch to 64, map a cheap transform, then prefetch so the next batch is ready before the model asks for it.

In [ ]:
def make_pipeline(x, y, batch_size=64, seed=42):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.shuffle(len(x), seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda img, lab: (tf.reshape(img, (-1, 784)), lab))
    return ds.prefetch(tf.data.AUTOTUNE)


train_ds = make_pipeline(x_train, y_train)
xb, yb = next(iter(train_ds))
print(xb.shape, yb.shape, len(train_ds), "batches per epoch")

## From scratch: a training loop with GradientTape

Write the loop yourself. The tape records the forward pass. `tape.gradient` returns one gradient per weight. The optimiser applies them. This is the loop `model.fit` hides.

In [ ]:
model = keras.Sequential(
    [keras.layers.Input((784,)), keras.layers.Dense(64, activation="relu"), keras.layers.Dense(10)]
)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
opt = keras.optimizers.SGD(learning_rate=0.1)


def train_epoch(ds):
    total, count = 0.0, 0
    for xb, yb in ds:
        with tf.GradientTape() as tape:
            loss = loss_fn(yb, model(xb, training=True))
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply(grads, model.trainable_variables)
        total += float(loss) * len(xb)
        count += len(xb)
    return total / count


losses = [train_epoch(train_ds) for _ in range(5)]
print([round(v, 3) for v in losses])

## Library version

The same model through `model.fit`, fed by the same pipeline. Keras runs the tape for you. The clone starts from fresh weights, so expect a small gap: 88.2% test accuracy for the tape loop and 87.2% for `fit`, one seed apart.

In [ ]:
test_ds = make_pipeline(x_test, y_test)
fit_model = keras.models.clone_model(model)
fit_model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.1), loss=loss_fn, metrics=["accuracy"]
)
fit_model.fit(train_ds, epochs=5, verbose=0)

tape_pred = model(x_test.reshape(-1, 784)).numpy().argmax(axis=1)
print("tape loop test accuracy:", round(float((tape_pred == y_test).mean()), 3))
print("fit test accuracy:", round(fit_model.evaluate(test_ds, verbose=0)[1], 3))

## Tuning and what breaks

Two things break pipelines in practice. Shuffling after batching only reorders whole batches, so every batch holds the same 64 neighbouring samples in every epoch. And a map step written in plain Python is slow and cannot be traced. The cell below measures the first problem. Each sample is its own index, so a full batch is a block of neighbours when its indices span exactly 63.

In [ ]:
ids = np.arange(1000)


def batch_spans(ds, epochs=3):
    return [int(np.ptp(next(iter(ds)).numpy())) for _ in range(epochs)]


bad = tf.data.Dataset.from_tensor_slices(ids).batch(64, drop_remainder=True).shuffle(10, seed=1)
good = tf.data.Dataset.from_tensor_slices(ids).shuffle(1000, seed=1).batch(64, drop_remainder=True)
print("shuffle after batch, index span of first batch:", batch_spans(bad))
print("shuffle before batch, index span of first batch:", batch_spans(good))

The cost of this lesson is small here and large on real data. Put the shuffle before the batch, and check one batch by eye before you train for an hour.